# 00 — Kurulum ve Kontrol

Bu notebook Colab ortamını hazırlar: GPU kontrolü, Drive bağlantısı, repo klonu, bağımlılık kurulumu ve duman testi (BM25 ile küçük bir erişim denemesi).

**Başlamadan önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU** seçin.

Hücreleri sırayla çalıştırın. Bir hücre hata verirse önce o hatayı çözün; sonraki hücreler öncekilere bağımlıdır.

In [ ]:
# 1) GPU ve ortam kontrolü
import sys
print('Python:', sys.version)
!nvidia-smi

In [ ]:
# 2) Google Drive'ı bağla (sonuçlar ve model önbelleği burada tutulur)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Repoyu klonla (varsa güncelle)
import os
REPO_URL = 'https://github.com/EmreK37/RAG-Tabanl-T-rk-e-Soru-Cevap-Sistemleri.git'
REPO_DIR = '/content/RAG-proje'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
!git log --oneline -3 || echo 'henüz commit yok'

In [ ]:
# 4) Bağımlılıkları kur (Colab'da torch hazır gelir; requirements bunu korur)
!pip install -q -r requirements.txt
print('kurulum tamam')

In [ ]:
# 5) Konfigürasyonu oku ve özetle
import sys
sys.path.insert(0, '/content/RAG-proje')
from src.config import load_config

cfg = load_config()
print('Deney:', cfg['deney']['ad'])
print('Tohumlar:', cfg['deney']['tohumlar'])
print('Modeller:')
for m in cfg['modeller']:
    print(' -', m['ad'], '->', m.get('hf'))

In [ ]:
# 6) Duman testi: yönetmeliği parçala + BM25 ile örnek sorgu
# (model indirmez, CPU ile saniyeler sürer)
from src.data import korpus_yukle, yonetmelik_parcala
from src.retrieval import Bm25Retriever

metin = korpus_yukle(cfg['veri']['korpus'])
chunklar = yonetmelik_parcala(
    metin,
    boyut=cfg['parcalama']['boyut_karakter'],
    ortusme=cfg['parcalama']['ortusme_karakter'],
)
print(f'Chunk sayısı: {len(chunklar)}')

bm25 = Bm25Retriever(chunklar)
sorgu = 'Öğrencinin mezuniyet koşulları nelerdir?'
for chunk_id, skor in bm25.retrieve(sorgu, k=3):
    print(f'{skor:8.2f}  {chunk_id}')
print('\nOrtam hazır. Sıradaki adım: 01_veri_hazirlama.ipynb')

## Sorun giderme

- **GPU yok görünüyorsa:** Çalışma zamanı türünü T4 GPU olarak değiştirip oturumu yeniden başlatın.
- **pip çakışması:** Hücreyi bir kez daha çalıştırın; çözülmezse hata çıktısını ekip ile paylaşın (kendi başınıza sürüm sabitlerini değiştirmeyin, `requirements.txt` üzerinden PR açın).
- **Drive dolu hatası:** Model önbelleğini Drive'a değil oturum diskine yazın; sonuç CSV'leri küçüktür, sorun çıkarmaz.